# 📈 Enterprise Profit Prediction & Multi-Model Tournament

This notebook provides end-to-end exploratory data analysis, feature engineering, a 10-model tournament cross-validation benchmark, and SHAP explainability for enterprise profit forecasting.

In [ ]:
import os
import json
import joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Candidate Models Zoo (10 Models)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

import shap
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Data Ingestion & Preprocessing
Load historical spend and profit figures from local raw data directory.

In [ ]:
data_path = Path('../data/raw/50_Startups.csv')
if not data_path.exists():
    data_path = Path('data/raw/50_Startups.csv')
if not data_path.exists():
    url = 'https://drive.google.com/uc?id=1Z7RKmScBO7n9vcDIG3Xeo853Ics4QFaF'
    df = pd.read_csv(url)
else:
    df = pd.read_csv(data_path)

print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
df.info()
df.describe()

## 2. Exploratory Data Analysis & Pairwise Correlation

In [ ]:
plt.figure(figsize=(8, 6))
cols = ['R&D Spend', 'Administration', 'Marketing Spend', 'Profit']
sns.heatmap(df[cols].corr(), annot=True, cmap='Blues', fmt='.3f')
plt.title('Correlation Matrix: Departmental Spend vs Profit')
plt.show()

In [ ]:
pairplot = sns.pairplot(df, hue='State' if 'State' in df.columns else None, palette='viridis')
pairplot.fig.suptitle('Multivariate Feature Relationships', y=1.02)
plt.show()

## 3. Feature Matrix & Train-Test Split

In [ ]:
features = ['R&D Spend', 'Administration', 'Marketing Spend']
X = df[features]
y = df['Profit']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f'Training set: {X_train.shape}, Test set: {X_test.shape}')

## 4. 10-Model Competitive Tournament Zoo
Benchmarking 10 candidate models with 5-Fold cross-validation.

In [ ]:
candidate_models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=100.0, random_state=42),
    'ElasticNet': ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=42),
    'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42),
    'Extra Trees': ExtraTreesRegressor(n_estimators=100, max_depth=6, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.08, random_state=42),
    'Support Vector Regressor': SVR(kernel='rbf', C=10000, epsilon=1000),
    'K-Nearest Neighbors': KNeighborsRegressor(n_neighbors=5, weights='distance'),
    'Bayesian Ridge': BayesianRidge()
}

# Build Voting Regressor ensemble meta-model
top_estimators = [
    ('rf', candidate_models['Random Forest']),
    ('gb', candidate_models['Gradient Boosting']),
    ('ridge', candidate_models['Ridge Regression'])
]
candidate_models['VotingRegressor (Ensemble)'] = VotingRegressor(estimators=top_estimators)

tournament_results = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in candidate_models.items():
    cv_r2 = cross_val_score(model, X_train_scaled, y_train, cv=kf, scoring='r2').mean()
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    test_r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    tournament_results.append({
        'Model': name,
        'CV R2 (5-Fold)': cv_r2,
        'Test R2': test_r2,
        'Test RMSE': rmse,
        'Test MAE': mae
    })

leaderboard_df = pd.DataFrame(tournament_results).sort_values('Test R2', ascending=False).reset_index(drop=True)
leaderboard_df

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=leaderboard_df, y='Model', x='Test R2', palette='viridis')
plt.title('Model Tournament Leaderboard: Out-of-Sample R2 Score')
plt.xlabel('R2 Score')
plt.xlim(0, 1.0)
plt.axvline(0.9, color='red', linestyle='--', label='Target R2 > 0.90')
plt.legend()
plt.show()

## 5. SHAP Explainability & Feature Attribution
Understand feature contributions using cooperative game-theory Shapley values.

In [ ]:
best_model = candidate_models['Random Forest']
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_scaled)

# SHAP Summary Plot
shap.summary_plot(shap_values, X_test_scaled, feature_names=features, show=False)
plt.title('SHAP Feature Importance & Impact Direction', y=1.05)
plt.show()

## 6. Model Artifact Serialization
Export best performing model, scaler, and tournament metadata for deployment in Streamlit and FastAPI.

In [ ]:
artifacts_dir = Path('../models/artifacts')
if not artifacts_dir.exists():
    artifacts_dir = Path('models/artifacts')
artifacts_dir.mkdir(parents=True, exist_ok=True)

# Export complete model payload
payload = {
    'models': candidate_models,
    'default_model': candidate_models['VotingRegressor (Ensemble)'],
    'best_model_name': 'VotingRegressor (Ensemble Meta-Model)'
}

joblib.dump(payload, artifacts_dir / 'model.pkl')
joblib.dump(scaler, artifacts_dir / 'scaler.pkl')
print('Model and Scaler artifacts successfully serialized.')